# Country Mappings

This notebook contains mappings for country codes found in the stocks dataset.

# Mappings

In [ ]:
# 1. Sectors
sectors = ['Communication Services', 'Consumer Discretionary', 'Consumer Staples', 
               'Energy', 'Financials', 'Health Care', 'Industrials', 
               'Information Technology', 'Materials', 'Real Estate', 'Utilities']

# 2. Industry Groups
industry_groups = [
    'Automobiles & Components',
    'Banks',
    'Capital Goods',
    'Commercial & Professional Services',
    'Consumer Durables & Apparel',
    'Consumer Services',
    'Energy',
    'Equity Real Estate Investment Trusts (REITs)',
    'Financial Services',
    'Food & Staples Retailing',
    'Food Beverage & Tobacco',
    'Health Care Equipment & Services',
    'Household & Personal Products',
    'Insurance',
    'Materials',
    'Media & Entertainment',
    'Pharmaceuticals, Biotechnology & Life Sciences',
    'Real Estate Management & Development',
    'Retailing',
    'Semiconductors & Semiconductor Equipment',
    'Software & Services',
    'Technology Hardware & Equipment',
    'Telecommunication Services',
    'Transportation',
    'Utilities'
]

# 3. Sectors to Industry Groups Mapping (GICS Structure)
sector_to_industry_groups = {
    'Communication Services': [
        'Media & Entertainment',
        'Telecommunication Services'
    ],
    'Consumer Discretionary': [
        'Automobiles & Components',
        'Consumer Durables & Apparel',
        'Consumer Services',
        'Retailing'
    ],
    'Consumer Staples': [
        'Food & Staples Retailing',
        'Food Beverage & Tobacco',
        'Household & Personal Products'
    ],
    'Energy': [
        'Energy'
    ],
    'Financials': [
        'Banks',
        'Financial Services',
        'Insurance'
    ],
    'Health Care': [
        'Health Care Equipment & Services',
        'Pharmaceuticals, Biotechnology & Life Sciences'
    ],
    'Industrials': [
        'Capital Goods',
        'Commercial & Professional Services',
        'Transportation'
    ],
    'Information Technology': [
        'Semiconductors & Semiconductor Equipment',
        'Software & Services',
        'Technology Hardware & Equipment'
    ],
    'Materials': [
        'Materials'
    ],
    'Real Estate': [
        'Equity Real Estate Investment Trusts (REITs)',
        'Real Estate Management & Development'
    ],
    'Utilities': [
        'Utilities'
    ]
}

# 4. Country Code to Country Name Mapping
country_code_to_name = {
    'AT': 'Austria',
    'AU': 'Australia',
    'BE': 'Belgium',
    'BM': 'Bermuda',
    'CA': 'Canada',
    'CH': 'Switzerland',
    'CL': 'Chile',
    'CN': 'China',
    'DE': 'Germany',
    'DK': 'Denmark',
    'ES': 'Spain',
    'FR': 'France',
    'GB': 'United Kingdom',
    'HK': 'Hong Kong',
    'IE': 'Ireland',
    'IL': 'Israel',
    'IN': 'India',
    'IT': 'Italy',
    'JP': 'Japan',
    'KR': 'South Korea',
    'LU': 'Luxembourg',
    'NL': 'Netherlands',
    'NO': 'Norway',
    'PA': 'Panama',
    'PE': 'Peru',
    'PT': 'Portugal',
    'SE': 'Sweden',
    'SG': 'Singapore',
    'US': 'United States',
    'ZA': 'South Africa'
}

# 5. Country Code to Region Mapping
region_to_countries = {
    'Europe': ['AT', 'BE', 'CH', 'DE', 'DK', 'ES', 'FR', 'GB', 'IE', 'IL', 'IT', 'LU', 'NL', 'NO', 'PT', 'SE'],
    'North America': ['BM', 'CA', 'PA', 'US'],
    'South America': ['CL', 'PE'],
    'Asia': ['CN', 'HK', 'IN', 'JP', 'KR', 'SG'],
    'Africa': ['ZA'],
    'Oceania': ['AU'],
    'World': ['AT', 'AU', 'BE', 'BM', 'CA', 'CH', 'CL', 'CN', 'DE', 'DK', 'ES', 'FR', 'GB', 'HK', 'IE', 'IL', 'IN', 'IT', 'JP', 'KR', 'LU', 'NL', 'NO', 'PA', 'PE', 'PT', 'SE', 'SG', 'US', 'ZA']
}

# 6. Market Classification to Country Codes Mapping
market_to_countries = {
    'Developed': ['AT', 'AU', 'BE', 'CA', 'CH', 'DE', 'DK', 'ES', 'FR', 'GB', 'IE', 'IL', 'IT', 'JP', 'LU', 'NL', 'NO', 'PT', 'SE', 'SG', 'US'],
    'Emerging': ['BM', 'CL', 'CN', 'HK', 'IN', 'KR', 'PA', 'PE', 'ZA'],
    'Investable Universe': ['AT', 'AU', 'BE', 'BM', 'CA', 'CH', 'CL', 'CN', 'DE', 'DK', 'ES', 'FR', 'GB', 'HK', 'IE', 'IL', 'IN', 'IT', 'JP', 'KR', 'LU', 'NL', 'NO', 'PA', 'PE', 'PT', 'SE', 'SG', 'US', 'ZA']
}

# Benchmark Calculation


In [ ]:
import pandas as pd
from typing import Literal, Optional

# Load the data with proper encoding
# --------------------------------------------------------------------------------
stocks_df = pd.read_csv('stocks sectors data/dataset_with_ISINs_as_primary_key.csv', encoding='latin-1')
sectors_df = pd.read_csv('stocks sectors data/Sector_Data.csv', encoding='latin-1')
industry_groups_df = pd.read_csv('stocks sectors data/Industry_Groups_Data.csv', encoding='latin-1')

# Merge the datasets on ISIN
merged_df = pd.merge(stocks_df, sectors_df[['ISIN', 'Communication Services', 'Consumer Discretionary', 
                                              'Consumer Staples', 'Energy', 'Financials', 'Health Care', 
                                              'Industrials', 'Information Technology', 'Materials', 
                                              'Real Estate', 'Utilities']], 
                      on='ISIN', how='left')

# Get actual industry group columns from CSV (excluding [u] suffixes and metadata columns)
industry_groups_csv_cols = [col for col in industry_groups_df.columns 
                            if col not in ['Description', 'ID', 'ISIN', 'Name', 'Parse Date', 'GICS Level', 'GICS IndustryGroup', 'confidence'] 
                            and not col.endswith('[u]')]

# Merge industry groups data - use suffixes to avoid column name conflicts
merged_df = pd.merge(merged_df, industry_groups_df[['ISIN'] + industry_groups_csv_cols], 
                      on='ISIN', how='left', suffixes=('', '_ig'))



# --------------------------------------------------------------------------------
def calc_benchmarks(level: Literal['country', 'region', 'market'] = 'country', code: Optional[str] = None):
    """
    Calculate the sector and industry group weights for a country, region, or market.
    
    Args:
        level (str): Level of aggregation - 'country', 'region', or 'market'
        code (str): 
            - For country: Two-letter country code (e.g., 'US', 'GB', 'DE')
            - For region: Region name (e.g., 'Europe', 'Asia', 'North America')
            - For market: Market classification ('Developed' or 'Emerging')
    
    Returns:
        dict: Dictionary with sector names as keys. Each value is a tuple with:
            - [0]: float (0-1) representing the sector's weight
            - [1]: dict with industry group names as keys and their weights (0-1) as values
              (only industry groups belonging to that sector according to GICS structure)
    """
    # Determine which countries to include based on level
    if level == 'country':
        if code is None:
            raise ValueError("Country code must be provided when level='country'")
        country_codes = [code]
    
    elif level == 'region':
        if code is None:
            raise ValueError("Region name must be provided when level='region'")
        if code not in region_to_countries:
            raise ValueError(f"Unknown region: {code}. Valid regions: {list(region_to_countries.keys())}")
        country_codes = region_to_countries[code]
    
    elif level == 'market':
        if code is None:
            raise ValueError("Market classification must be provided when level='market'")
        if code not in market_to_countries:
            raise ValueError(f"Unknown market: {code}. Valid markets: {list(market_to_countries.keys())}")
        country_codes = market_to_countries[code]
    
    else:
        raise ValueError(f"Invalid level: {level}. Must be 'country', 'region', or 'market'")
    
    # Get all firms from the specified countries
    selected_firms = merged_df[merged_df['country'].isin(country_codes)].copy()
    if len(selected_firms) == 0:
        return {'sectors': {}, 'industry_groups': {}}
    
    # Calculate total market cap
    total_market_cap = selected_firms['mktCap (EUR)'].sum()
    if total_market_cap == 0:
        return {'sectors': {}, 'industry_groups': {}}
    
    # Calculate industry group weights first - initialize all industry groups to 0.0
    all_industry_group_weights = {}
    
    for industry_group in industry_groups_csv_cols:
        # For each firm: firm_industry_group_weight * (firm_market_cap / total_market_cap)
        # Sum this across all firms
        selected_firms['weighted_contribution'] = (
            selected_firms[industry_group].fillna(0) / 100.0 *  # Convert percentage to decimal
            selected_firms['mktCap (EUR)'] / total_market_cap
        )
        ig_weight = selected_firms['weighted_contribution'].sum()
        all_industry_group_weights[industry_group] = ig_weight
    
    # Build result structure with sectors as top-level keys
    result = {}
    
    for sector in sectors:
        # Calculate sector weight
        selected_firms['weighted_contribution'] = (
            selected_firms[sector].fillna(0) / 100.0 *
            selected_firms['mktCap (EUR)'] / total_market_cap
        )
        sector_weight = selected_firms['weighted_contribution'].sum()
        
        # Get industry groups for this sector from the mapping
        sector_industry_groups = sector_to_industry_groups.get(sector, [])
        
        # Build industry groups dict for this sector
        industry_groups_dict = {}
        for ig in sector_industry_groups:
            # Find matching column name in CSV (handle naming differences)
            matched_weight = 0.0
            for csv_col in industry_groups_csv_cols:
                # Check for exact match or partial match
                if ig == csv_col or ig in csv_col or csv_col in ig:
                    matched_weight = all_industry_group_weights.get(csv_col, 0.0)
                    break
            
            industry_groups_dict[ig] = round(matched_weight, 4)
        
        result[sector] = (round(sector_weight, 4), industry_groups_dict)
    
    return result

# Convenience functions for backwards compatibility and easier usage
def calc_country_benchmarks(country_code):
    """Calculate sector weights for a specific country."""
    return calc_benchmarks(level='country', code=country_code)

def calc_region_benchmarks(region_name):
    """Calculate sector weights for a specific region."""
    return calc_benchmarks(level='region', code=region_name)

def calc_market_benchmarks(market_classification):
    """Calculate sector weights for a specific market (Developed/Emerging)."""
    return calc_benchmarks(level='market', code=market_classification)


In [36]:
# Test the benchmark function
result = calc_country_benchmarks("US")
print(result)

# Check if weights sum to 1
sector_sum = sum(sector_data[0] for sector_data in result.values())

# Calculate total industry group weight across all sectors
all_ig_weights = []
for sector_data in result.values():
    all_ig_weights.extend(sector_data[1].values())
industry_group_sum = sum(all_ig_weights)

print(f"Sector weights sum: {sector_sum:.6f}")
print(f"Industry group weights sum: {industry_group_sum:.6f}")

{'Communication Services': (np.float64(0.1316), {'Media & Entertainment': np.float64(0.1096), 'Telecommunication Services': np.float64(0.0079)}), 'Consumer Discretionary': (np.float64(0.1492), {'Automobiles & Components': np.float64(0.0174), 'Consumer Durables & Apparel': np.float64(0.0157), 'Consumer Services': np.float64(0.0294), 'Retailing': np.float64(0.0183)}), 'Consumer Staples': (np.float64(0.0389), {'Food & Staples Retailing': np.float64(0.0183), 'Food Beverage & Tobacco': np.float64(0.0117), 'Household & Personal Products': np.float64(0.0087)}), 'Energy': (np.float64(0.0272), {'Energy': np.float64(0.0272)}), 'Financials': (np.float64(0.0861), {'Banks': np.float64(0.0283), 'Financial Services': np.float64(0.0415), 'Insurance': np.float64(0.0154)}), 'Health Care': (np.float64(0.0785), {'Health Care Equipment & Services': np.float64(0.0382), 'Pharmaceuticals, Biotechnology & Life Sciences': np.float64(0.0401)}), 'Industrials': (np.float64(0.0626), {'Capital Goods': np.float64(0.0

In [37]:
# Show full structure for one country
print("Full benchmark structure for US:")
print("="*70)
for sector, (sector_weight, industry_groups) in result.items():
    if sector_weight > 0:
        print(f"\n{sector}: {sector_weight:.4f}")
        if industry_groups:
            for ig, weight in industry_groups.items():
                print(f"  - {ig}: {weight:.4f}")

Full benchmark structure for US:

Communication Services: 0.1316
  - Media & Entertainment: 0.1096
  - Telecommunication Services: 0.0079

Consumer Discretionary: 0.1492
  - Automobiles & Components: 0.0174
  - Consumer Durables & Apparel: 0.0157
  - Consumer Services: 0.0294
  - Retailing: 0.0183

Consumer Staples: 0.0389
  - Food & Staples Retailing: 0.0183
  - Food Beverage & Tobacco: 0.0117
  - Household & Personal Products: 0.0087

Energy: 0.0272
  - Energy: 0.0272

Financials: 0.0861
  - Banks: 0.0283
  - Financial Services: 0.0415
  - Insurance: 0.0154

Health Care: 0.0785
  - Health Care Equipment & Services: 0.0382
  - Pharmaceuticals, Biotechnology & Life Sciences: 0.0401

Industrials: 0.0626
  - Capital Goods: 0.0386
  - Commercial & Professional Services: 0.0106
  - Transportation: 0.0082

Information Technology: 0.3880
  - Semiconductors & Semiconductor Equipment: 0.2154
  - Software & Services: 0.1317
  - Technology Hardware & Equipment: 0.0081

Materials: 0.0128
  - Mate